# Tareas

## 1. Diseña una base de datos Cassandra para dar servicio a las lecturas y escrituras anteriores. Argumenta tus decisiones de diseño.

En Cassandra no se diseña “por entidades y relaciones”, sino por consultas. El enunciado da 3 lecturas y 2 escrituras, así que el diseño va a salir directamente de ahí.

Lo que vamos a hacer es es desnormalizar y crear tablas específicas para cada patrón de acceso, en vez de intentar reproducir el modelo relacional.

### Tablas que diseñaríamos:

#### Tabla del perfil del usuario

Hemos decidido incluir una tabla conceptual llamada `user_by_email` con el objetivo de poder resolver de forma rápida la información básica de cada jugador a partir de su email. Esta decisión la hemos tomado porque, al analizar las operaciones del sistema, vimos que las escrituras llegan utilizando el email como dato principal, mientras que algunas lecturas necesitan recuperar campos como el `user_id`. Por tanto, nos interesa que esa conversión entre email e identidad del jugador sea directa y eficiente.

La clave primaria de esta tabla entonces será `email`, que actuará como **partition key**. 
Además, la tabla contendrá las columnas `email`, `user_id`, `user_name` y `country`. Hemos escogido exactamente estos campos porque son los que necesitamos reutilizar después en distintas consultas del sistema. Por ejemplo, en el **Hall of Fame** necesitamos mostrar el `user_name`, en el **Top Horde** necesitamos `user_id`, `user_name` y `email` y además, sabemos que los rankings son locales por país, lo que también supone qu necesitamos obtener el `country` de forma inmediata.

Aunque esta tabla no representa un leaderboard como tal, en el diseño de Cassandra nos parece fundamental porque evita tener que hacer joins entre tablas. En lugar de depender de relaciones complejas en tiempo de consulta, preferimos desnormalizar la información y dejar preparado un acceso directo a los datos más importantes del usuario.

#### Tabla de catálogo de mazmorras

Hemos decidido incluir una tabla conceptual llamada `dungeon_by_id` con el objetivo de poder traducir de forma directa cada `dungeon_id` a su correspondiente `dungeon_name`. Esta decisión la hemos tomado porque, al revisar las consultas que debe soportar el sistema, vimos que no bastaba con almacenar únicamente el identificador de la mazmorra, sino que también necesitábamos poder  disponer de su nombre de forma inmediata, para poder devolver la información completa que exige la aplicación.

La clave primaria de esta tabla será `dungeon_id`, que actuará como identificador único de cada mazmorra. Además, la tabla contendrá las columnas `dungeon_id` y `dungeon_name`. Hemos escogido esta estructura porque es suficiente para resolver exactamente la necesidad que tenemos, sin añadir información innecesaria y manteniendo un acceso muy simple y eficiente.

Aunque esta tabla tampoco representa un leaderboard en sí, nos parece necesaria dentro del diseño general porque permite recuperar el nombre de la mazmorra sin depender de operaciones adicionales. En concreto, en el **Hall of Fame** se devuelve tanto el `dungeon_id` como el `dungeon_name`, por lo que necesitamos que esa traducción pueda hacerse de forma directa y rápida.

#### Tabla para estadísticas de usuario por mazmorra

Hemos decidido incluir una tabla conceptual llamada `runs_by_user_dungeon` para responder a la consulta que recibe como entrada `user_id` y `dungeon_id` y debe devolver una lista de partidas ordenada por menor tiempo, con los campos `{time_minutes, date}`.

La clave primaria estará formada por `((user_id, dungeon_id), time_minutes, date)`. Es decir, usaremos `user_id` y `dungeon_id` como **partition key** compuesta, y `time_minutes` y `date` como **clustering keys**, ambas en orden ascendente. De esta forma, cada partición agrupa las runs de un usuario en una mazmorra concreta, y dentro de ella los resultados quedan ya ordenados por menor tiempo.

Hemos elegido este diseño porque la consulta siempre fija `user_id` y `dungeon_id`, y además necesita que la salida aparezca ordenada por tiempo. Así, Cassandra puede devolver directamente los resultados en el orden que pide el leaderboard, sin necesidad de hacer ordenaciones adicionales en lectura.

#### Tabla para Hall of Fame por país y mazmorra

Hemos decidido utilizar la tabla conceptual `hall_of_fame_by_country_dungeon` para cubrir la lectura del **Hall of Fame**. Esta consulta recibe como entrada un `country` y debe devolver, para cada mazmorra del juego, el **Top 5** de jugadores más rápidos de ese país, incluyendo `email`, `user_name`, `time_minutes` y `date`.

La clave primaria será `((country, dungeon_id), time_minutes, date, email)`. Es decir, usaremos `country` y `dungeon_id` como **partition key** compuesta, y `time_minutes`, `date` y `email` como **clustering keys** en orden ascendente. Además, la tabla contendrá las columnas `country`, `dungeon_id`, `dungeon_name`, `email`, `user_name`, `time_minutes` y `date`.

Hemos elegido este diseño porque permite una lectura muy eficiente por mazmorra. Para servir la consulta, el servidor obtiene primero la lista de mazmorras y después consulta cada partición `(country, dungeon_id)` con `LIMIT 5`. Con esas respuestas construye el JSON final del leaderboard.

Nos parece la mejor opción porque la escritura sigue siendo sencilla, ya que al terminar una dungeon basta con insertar una nueva fila. Además, evitamos mantener listas materializadas o estructuras agregadas más difíciles de actualizar. Aunque esta lectura no sale de una sola query de Cassandra, sino de varias, una por mazmorra, lo consideramos asumible porque el número de mazmorras será finito y controlado y porque este ranking no es una parte crítica del gameplay.

También valoramos como alternativa guardar directamente el Top 5 materializado, pero no la hemos escogido como diseño principal porque obligaría a recalcular y mantener ese top en cada escritura, complicando bastante más la lógica de actualización.

#### Tabla de contador de muertes por jugador en una horda

Hemos decidido incluir una tabla conceptual llamada `horde_kills_by_country_event_user` para almacenar el acumulado de kills de cada jugador dentro de una horda concreta. Esta tabla parte de que la escritura llega con `event_id`, `email` y `monster_id`, mientras que la lectura del **Top Horde** necesita obtener, para un `country`, un `event_id` y un valor `K`, los jugadores con más kills.

La clave primaria será `((country, event_id), user_id)`. Es decir, usaremos `country` y `event_id` como **partition key** compuesta, y `user_id` como **clustering key**. Además, la tabla contendrá las columnas `country`, `event_id`, `user_id`, `email`, `user_name` y `n_killed`.

Hemos elegido este diseño porque cada evento de tipo “user kills monster” debe actualizar el contador de kills del jugador dentro de esa horda concreta, y esta tabla nos permite guardar ese acumulado de forma directa.

Aun así, esta tabla por sí sola no resuelve la lectura del ranking, ya que aunque `n_killed` se actualice correctamente como columna, Cassandra no permite obtener de forma eficiente el top K ordenado por ese valor si no forma parte de la clave. Por eso, hemos concluido que esta tabla es necesaria para acumular la información, pero debe complementarse con otra tabla orientada específicamente al ranking.

#### Tabla de ranking de Horde ordenada por número de kills

Hemos decidido incluir una tabla conceptual llamada `top_horde_by_country_event` para responder de forma eficiente a la lectura del **Top Horde**. Esta consulta recibe como entrada `country`, `event_id` y `K`, y debe devolver los `K` jugadores con mayor número de kills en esa horda.

La clave primaria será `((country, event_id), n_killed, user_id)`. Es decir, usaremos `country` y `event_id` como **partition key** compuesta, y `n_killed` y `user_id` como **clustering keys**. El orden de clustering será `n_killed DESC` y `user_id ASC`. Además, la tabla contendrá las columnas `country`, `event_id`, `user_id`, `user_name`, `email` y `n_killed`.

Hemos elegido este diseño porque encaja de forma muy natural con la lectura que pide el sistema. Al consultar la partición `(country, event_id)` y aplicar `LIMIT K`, Cassandra ya devuelve directamente los jugadores ordenados de mayor a menor número de kills, que es exactamente lo que necesita el leaderboard.

---

## 2. Crea las consultas .sql necesarias para exportar los datos de la base de datos relacional a ficheros .csv. Los ficheros deberán tener un formato acorde al diseño del punto 1.

In [ ]:
#!pip install mysql-connector-python pandas

In [1]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
password = os.getenv("MY_PASSWORD")

abrimos la conexion con la base de datos que hemos creado en MySQL Workbench

In [3]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="videogame")

In [4]:
show_tables="SHOW TABLES"
tables = pd.read_sql(show_tables, conn)
print(tables)

   Tables_in_videogame
0    completeddungeons
1         containsloot
2      containsmonster
3              dungeon
4                event
5                hints
6                kills
7                 loot
8              monster
9                 room
10      roomsconnected
11             webuser


C:\Users\adrig\AppData\Local\Temp\ipykernel_27772\3296000829.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql(show_tables, conn)


y ahora empezamos a ejecutar nuestras querys en relacion a cómo hemos definido que va a ser nuestra estructura

In [23]:
query_user_by_email = """SELECT
                email,
                ROW_NUMBER() OVER (ORDER BY email) AS user_id,
                userName AS user_name,
                country
            FROM webuser"""

lo guardamos en un csv para luego poder trabahar con ello

In [24]:
df_user_by_email = pd.read_sql(query_user_by_email, conn)

print(df_user_by_email.head())
print(df_user_by_email.shape)

                    email  user_id     user_name country
0        aabe@example.net        1  minorusuzuki   ja_JP
1       aaoki@example.com        2  rikamurakami   ja_JP
2     aaron03@example.com        3       robin84   en_US
3  aarongreen@example.com        4    leejessica   en_US
4      aaubry@example.net        5   catherine30   fr_FR
(9838, 4)


C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\1477136088.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_user_by_email = pd.read_sql(query_user_by_email, conn)


In [26]:
df_user_by_email.to_csv("data/user_by_email.csv", index=False)

In [70]:
df_user_by_email.isna().sum()   

email        0
user_id      0
user_name    0
country      0
dtype: int64

In [27]:
query_dungeon_by_id = """SELECT
                            idD AS dungeon_id,
                            name AS dungeon_name
                        FROM dungeon"""

df_dungeon_by_id = pd.read_sql(query_dungeon_by_id, conn)

print(df_dungeon_by_id.head())
print(df_dungeon_by_id.shape)

df_dungeon_by_id.to_csv("data/dungeon_by_id.csv", index=False)

   dungeon_id                                       dungeon_name
0           0             Burghap, Prison of the Jealous Hippies
1           1  Burgstream, Culverts of the Bashful Sumo Wrest...
2           2       Shadysparth, Hobble of the Wandering Stoners
3           3            Wanton, Culverts of the Jealous Thieves
4           4        Withystream, Castle of the Screeching Otaku
(20, 2)


C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\2901154859.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dungeon_by_id = pd.read_sql(query_dungeon_by_id, conn)


para la siguiente hay un detalle importante. como `user_id` no existe en la base relacional y lo estamos fabricando con `ROW_NUMBER()`, en esta query tenemos que reconstruir exactamente ese mismo mapeo y luego unirlo con la tabla donde estén las runs de mazmorra.

In [28]:
query_completeddungeons_preview = """
DESCRIBE completeddungeons
"""

df_completeddungeons_preview = pd.read_sql(query_completeddungeons_preview, conn)
print(df_completeddungeons_preview)

   Field          Type Null  Key Default           Extra
0   idCD           int   NO  PRI    None  auto_increment
1  email  varchar(100)   NO  MUL    None                
2    idD           int   NO  MUL    None                
3   date      datetime   NO         None                
4   time           int   NO         None                


C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\3653525880.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_completeddungeons_preview = pd.read_sql(query_completeddungeons_preview, conn)


In [29]:
query_runs_by_user_dungeon = """WITH user_map AS (
                                    SELECT
                                        email,
                                        ROW_NUMBER() OVER (ORDER BY email) AS user_id
                                    FROM webuser)
                                SELECT
                                    um.user_id,
                                    cd.idD AS dungeon_id,
                                    cd.time AS time_minutes,
                                    cd.date AS date
                                FROM completeddungeons cd
                                JOIN user_map um
                                    ON cd.email = um.email
                                ORDER BY um.user_id, cd.idD, cd.time, cd.date"""


df_runs_by_user_dungeon = pd.read_sql(query_runs_by_user_dungeon, conn)

C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\3952511678.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_runs_by_user_dungeon = pd.read_sql(query_runs_by_user_dungeon, conn)


In [30]:
print(df_runs_by_user_dungeon.head())
print(df_runs_by_user_dungeon.shape)

   user_id  dungeon_id  time_minutes                date
0        1           0             6 2022-04-11 10:43:14
1        1           0            19 2021-01-18 18:50:53
2        1           0            20 2020-04-18 04:46:20
3        1           0            20 2021-06-21 05:53:27
4        1           0            25 2021-11-03 11:29:36
(2000000, 4)


In [62]:
df_runs_by_user_dungeon.to_csv("data/runs_by_user_dungeon.csv", index=False)

In [69]:
len(df_runs_by_user_dungeon)

2000000

Hemos construido la consulta de `runs_by_user_dungeon` a partir de la tabla relacional `completeddungeons`, que almacena las mazmorras completadas por cada usuario junto con la fecha y el tiempo empleado. Como en el modelo relacional no existe un `user_id` explícito, primero generamos una correspondencia entre `email` y `user_id` mediante `ROW_NUMBER()`, manteniendo el mismo identificador usado en la exportación de usuarios. Después unimos esa correspondencia con `completeddungeons` por `email` y seleccionamos las columnas necesarias para Cassandra: `user_id`, `dungeon_id`, `time_minutes` y `date`. La consulta funciona porque adapta los datos relacionales al diseño de Cassandra y deja los resultados alineados con la clave de clustering de esta tabla

In [31]:
query_hall_of_fame_by_country_dungeon = """ SELECT
                                                wu.country AS country,
                                                cd.idD AS dungeon_id,
                                                d.name AS dungeon_name,
                                                cd.email AS email,
                                                wu.userName AS user_name,
                                                cd.time AS time_minutes,
                                                cd.date AS date
                                            FROM completeddungeons cd
                                            JOIN webuser wu
                                                ON cd.email = wu.email
                                            JOIN dungeon d
                                                ON cd.idD = d.idD
                                            ORDER BY wu.country, cd.idD, cd.time, cd.date, cd.email"""


df_hall_of_fame_by_country_dungeon = pd.read_sql(query_hall_of_fame_by_country_dungeon, conn)

C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\3026623043.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hall_of_fame_by_country_dungeon = pd.read_sql(query_hall_of_fame_by_country_dungeon, conn)


In [32]:
print(df_hall_of_fame_by_country_dungeon.head())
print(df_hall_of_fame_by_country_dungeon.shape)

df_hall_of_fame_by_country_dungeon.to_csv( "data/hall_of_fame_by_country_dungeon.csv", index=False)

  country  dungeon_id                            dungeon_name  \
0   de_DE           0  Burghap, Prison of the Jealous Hippies   
1   de_DE           0  Burghap, Prison of the Jealous Hippies   
2   de_DE           0  Burghap, Prison of the Jealous Hippies   
3   de_DE           0  Burghap, Prison of the Jealous Hippies   
4   de_DE           0  Burghap, Prison of the Jealous Hippies   

                             email        user_name  time_minutes  \
0               thea58@example.net        naseraziz             0   
1         scholtzcilly@example.org        zschleich             0   
2          steyradmila@example.com        krosemann             1   
3  neuschaefergabriela@example.org  schmiedeckerose             1   
4       noackveronique@example.org         zpeukert             1   

                 date  
0 2014-06-18 06:32:28  
1 2018-04-16 13:54:19  
2 2015-01-15 00:07:55  
3 2017-04-22 02:29:03  
4 2020-09-01 07:55:20  
(2000000, 7)


Hemos construido la consulta de `hall_of_fame_by_country_dungeon `a partir de la tabla `completeddungeons`, que contiene las runs realizadas por los jugadores, y la hemos enriquecido uniendo `webuser` y `dungeon`. De esta forma obtenemos, para cada partida, no solo el tiempo y la fecha, sino también el país del jugador, su nombre de usuario y el nombre de la mazmorra.

In [34]:
query_horde_kills_by_country_event_user = """WITH user_map AS (
                                                SELECT
                                                    email,
                                                    ROW_NUMBER() OVER (ORDER BY email) AS user_id
                                                FROM webuser
                                            )
                                            SELECT
                                                wu.country AS country,
                                                k.idE AS event_id,
                                                um.user_id AS user_id,
                                                k.email AS email,
                                                wu.userName AS user_name,
                                                COUNT(*) AS n_killed
                                            FROM kills k
                                            JOIN webuser wu
                                                ON k.email = wu.email
                                            JOIN user_map um
                                                ON k.email = um.email
                                            GROUP BY
                                                wu.country,
                                                k.idE,
                                                um.user_id,
                                                k.email,
                                                wu.userName
                                            ORDER BY
                                                wu.country,
                                                k.idE,
                                                um.user_id"""
                                            

df_horde_kills_by_country_event_user = pd.read_sql(query_horde_kills_by_country_event_user, conn)

C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\1949329457.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_horde_kills_by_country_event_user = pd.read_sql(query_horde_kills_by_country_event_user, conn)


In [35]:
print(df_horde_kills_by_country_event_user.head())
print(df_horde_kills_by_country_event_user.shape)

  country  event_id  user_id                       email         user_name  \
0   de_DE         0       26          abutte@example.org        haasefredi   
1   de_DE         0       35  adalbertdehmel@example.com           okeudel   
2   de_DE         0       50  adelbertspiess@example.net  misichergotthold   
3   de_DE         0       56          adietz@example.org   charlottejunken   
4   de_DE         0       57         adolf55@example.com          khettner   

   n_killed  
0         8  
1         7  
2         6  
3         7  
4         4  
(65081, 6)


In [36]:
df_horde_kills_by_country_event_user.to_csv("data/horde_kills_by_country_event_user.csv", index=False)

In [37]:
query_top_horde_by_country_event = """WITH user_map AS (
                                            SELECT
                                                email,
                                                ROW_NUMBER() OVER (ORDER BY email) AS user_id
                                            FROM webuser),

                                        kills_agg AS (
                                            SELECT
                                                wu.country AS country,
                                                k.idE AS event_id,
                                                um.user_id AS user_id,
                                                wu.userName AS user_name,
                                                k.email AS email,
                                                COUNT(*) AS n_killed
                                            FROM kills k
                                            JOIN webuser wu
                                                ON k.email = wu.email
                                            JOIN user_map um
                                                ON k.email = um.email
                                            GROUP BY
                                                wu.country,
                                                k.idE,
                                                um.user_id,
                                                wu.userName,
                                                k.email)
                                        SELECT
                                            country,
                                            event_id,
                                            user_id,
                                            user_name,
                                            email,
                                            n_killed
                                        FROM kills_agg
                                        ORDER BY country, event_id, n_killed DESC, user_id ASC """
                                       

df_top_horde_by_country_event = pd.read_sql(query_top_horde_by_country_event, conn)


C:\Users\adrig\AppData\Local\Temp\ipykernel_24060\126103411.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_horde_by_country_event = pd.read_sql(query_top_horde_by_country_event, conn)


In [38]:
print(df_top_horde_by_country_event.head())
print(df_top_horde_by_country_event.shape)


  country  event_id  user_id          user_name                         email  \
0   de_DE         0     9235     hartmanngloria          xullrich@example.org   
1   de_DE         0     7772    bernhardmangold  silvesterconradi@example.net   
2   de_DE         0     2896  johannekruschwitz     gretelholsten@example.com   
3   de_DE         0     6611          zdravko34            pero13@example.net   
4   de_DE         0     7253  oderwaldgabriella  roehrichtpaulina@example.org   

   n_killed  
0        19  
1        15  
2        14  
3        14  
4        14  
(65081, 6)


In [39]:
df_top_horde_by_country_event.to_csv("data/top_horde_by_country_event.csv",index=False)

---

## 3. Prepara un clúster local de 3 nodos todos en el mismo rack y datacenter. 

```powershell

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker compose down -v
[+] down 4/4
 ✔ Container cassandra3                Removed                                                                                                          1.9s
 ✔ Container cassandra2                Removed                                                                                                          1.9s
 ✔ Container cassandra1                Removed                                                                                                          1.9s
 ✔ Network cassandra_cluster_cassandra Removed                                                                                                          0.3s
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker pull cassandra:5.0
5.0: Pulling from library/cassandra
Digest: sha256:f56458c007cf111c663a23bd19d8e49a4228212b02a46c080f5384ded47ade6f
Status: Image is up to date for cassandra:5.0
docker.io/library/cassandra:5.0
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker compose up -d
[+] up 4/4
 ✔ Network cassandra_cluster_cassandra Created                                                                                                        0.1sss
 ✔ Container cassandra1                Healthy                                                                                                        96.0ss
 ✔ Container cassandra2                Healthy                                                                                                        130.7s
 ✔ Container cassandra3                Created                                                                                                        0.1sss
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker ps
CONTAINER ID   IMAGE           COMMAND                  CREATED         STATUS                             PORTS                                         NAMES
64178761632f   cassandra:5.0   "docker-entrypoint.s…"   2 minutes ago   Up 13 seconds (health: starting)   0.0.0.0:9044->9042/tcp, [::]:9044->9042/tcp   cassandra3
a4359c68bc28   cassandra:5.0   "docker-entrypoint.s…"   2 minutes ago   Up 48 seconds (healthy)            0.0.0.0:9043->9042/tcp, [::]:9043->9042/tcp   cassandra2
6b6776945b08   cassandra:5.0   "docker-entrypoint.s…"   2 minutes ago   Up 2 minutes (healthy)             0.0.0.0:9042->9042/tcp, [::]:9042->9042/tcp   cassandra1
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec -it cassandra1 nodetool status
Datacenter: DC1
===============
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack
UN  172.22.0.2  728.96 KiB  128     70.3%             3a67c8ec-39d8-4dc2-9036-6e54a0c69155  RAC1
UN  172.22.0.3  741.67 KiB  128     68.6%             05880a43-c804-4a65-b5c2-b696c16d34d7  RAC1
UN  172.22.0.4  714.53 KiB  128     61.2%             dec68c37-de4e-4b57-98e9-bd01c843375d  RAC1

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec -it cassandra1 ls /data
dungeon_by_id.csv  hall_of_fame_by_country_dungeon.csv  horde_kills_by_country_event_user.csv  top_horde_by_country_event.csv  user_by_email.csv
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec -it cassandra1 cqlsh
Connected to videogame_cluster at 127.0.0.1:9042
[cqlsh 6.2.0 | Cassandra 5.0.6 | CQL spec 3.4.7 | Native protocol v5]
Use HELP for help.
cqlsh>

Se ha desplegado un clúster local de Cassandra compuesto por 3 nodos mediante Docker Compose. Todos los nodos pertenecen al mismo datacenter (DC1) y al mismo rack (RAC1), tal y como exigía el enunciado. La correcta creación del clúster se ha verificado con nodetool status, comprobando que los tres nodos aparecen en estado UN (Up/Normal).

El archivo de `docker-compose.yml` está definido y localizado en la carperta **cassandra_cluster**

para borrar cualquier intento previo 

- PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> **docker compose down -v**

nos descargamos la imagen que vamos a usar de cassandra, en este caso la imagen `cassandra:5.0`

- PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> **docker pull cassandra:5.0**

creamos el docker compose abriendo la terminal en la misma carpeta que tenemos el archivo .yml

- PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> **docker compose up -d**

esperamos

listamos los contenedores y vemos que se han creado bien

- PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> **docker ps**

vemos el estado del cluster en una terminal iterativa

- PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> **docker exec -it cassandra1 nodetool status**

y viendo que esta todo en regla, podemos meternos en el cluster
- PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> **docker exec -it cassandra1 cqlsh**


---

## 4. Haz un fichero .cql que creen tu diseño en Cassandra y cargue los ficheros .csv creados en el paso 2. Se debe utilizar un factor de replicación de 2 y tener en cuenta que se las pruebas se ejecutaran en un clúster local. 

#### hacemos una pequeña prueba en la terminal


```powershell 
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec -it cassandra1 ls /data
dungeon_by_id.csv  hall_of_fame_by_country_dungeon.csv  horde_kills_by_country_event_user.csv  top_horde_by_country_event.csv  user_by_email.csv
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec -it cassandra1 cqlsh
Connected to videogame_cluster at 127.0.0.1:9042
[cqlsh 6.2.0 | Cassandra 5.0.6 | CQL spec 3.4.7 | Native protocol v5]
Use HELP for help.
cqlsh> CREATE KEYSPACE IF NOT EXISTS videogame
   ... WITH replication = {
   ...   'class': 'NetworkTopologyStrategy',
   ...   'DC1': 2
   ... };
cqlsh> USE videogame;
cqlsh:videogame> CREATE TABLE IF NOT EXISTS user_by_email (
             ...   email text,
             ...   user_id int,
             ...   user_name text,
             ...   country text,
             ...   PRIMARY KEY (email));

cqlsh:videogame> COPY user_by_email (email, user_id, user_name, country)
             ... FROM '/data/user_by_email.csv'
             ... WITH HEADER = TRUE;
Using 15 child processes

Starting copy of videogame.user_by_email with columns [email, user_id, user_name, country].
Processed: 9838 rows; Rate:    4494 rows/s; Avg. rate:    5352 rows/s
9838 rows imported from 1 files in 0 day, 0 hour, 0 minute, and 1.838 seconds (0 skipped).
cqlsh:videogame> SELECT * FROM user_by_email LIMIT 5;

 email                         | country | user_id | user_name
-------------------------------+---------+---------+------------------
 caldeiraana-laura@example.net |   pt_BR |     956 |           gramos
              dito@example.net |   ja_JP |    1658 |         shohei65
     hda-conceicao@example.com |   pt_BR |    3178 | fernandesgustavo
                bo@example.com |   ko_KR |     798 |          jaehogu
           mingqin@example.com |   zh_CN |    5659 |            minyi

(5 rows)
cqlsh:videogame>

In [5]:
import os
print(os.getcwd())
print(os.path.exists("load_cassandra.cql"))

c:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra
True


In [ ]:
base_content = """
CREATE KEYSPACE IF NOT EXISTS videogame
WITH replication = {
  'class': 'NetworkTopologyStrategy',
  'DC1': 2
};

USE videogame;
"""

In [48]:
content_user_by_email = """
CREATE TABLE IF NOT EXISTS user_by_email (
    email text,
    user_id int,
    user_name text,
    country text,
    PRIMARY KEY (email)
);

COPY user_by_email (email, user_id, user_name, country)
FROM '/data/user_by_email.csv'
WITH HEADER = TRUE;

"""

In [50]:
content_dungeon_by_id = """
CREATE TABLE IF NOT EXISTS dungeon_by_id (
    dungeon_id int,
    dungeon_name text,
    PRIMARY KEY (dungeon_id)
);

COPY dungeon_by_id (dungeon_id, dungeon_name)
FROM '/data/dungeon_by_id.csv'
WITH HEADER = TRUE;

"""

In [51]:
content_runs_by_user_dungeon = """
CREATE TABLE IF NOT EXISTS runs_by_user_dungeon (
    user_id int,
    dungeon_id int,
    time_minutes int,
    date timestamp,
    PRIMARY KEY ((user_id, dungeon_id), time_minutes, date)
) WITH CLUSTERING ORDER BY (time_minutes ASC, date ASC);

COPY runs_by_user_dungeon (user_id, dungeon_id, time_minutes, date)
FROM '/data/runs_by_user_dungeon.csv'
WITH HEADER = TRUE;

"""

In [52]:
content_hall_of_fame_by_country_dungeon = """
CREATE TABLE IF NOT EXISTS hall_of_fame_by_country_dungeon (
    country text,
    dungeon_id int,
    dungeon_name text,
    email text,
    user_name text,
    time_minutes int,
    date timestamp,
    PRIMARY KEY ((country, dungeon_id), time_minutes, date, email)
) WITH CLUSTERING ORDER BY (time_minutes ASC, date ASC, email ASC);

COPY hall_of_fame_by_country_dungeon (
    country, dungeon_id, dungeon_name, email, user_name, time_minutes, date
)
FROM '/data/hall_of_fame_by_country_dungeon.csv'
WITH HEADER = TRUE;

"""

In [53]:
content_horde_kills_by_country_event_user = """
CREATE TABLE IF NOT EXISTS horde_kills_by_country_event_user (
    country text,
    event_id int,
    user_id int,
    email text,
    user_name text,
    n_killed int,
    PRIMARY KEY ((country, event_id), user_id)
);

COPY horde_kills_by_country_event_user (
    country, event_id, user_id, email, user_name, n_killed
)
FROM '/data/horde_kills_by_country_event_user.csv'
WITH HEADER = TRUE;

"""

In [54]:
content_top_horde_by_country_event = """
CREATE TABLE IF NOT EXISTS top_horde_by_country_event (
    country text,
    event_id int,
    user_id int,
    user_name text,
    email text,
    n_killed int,
    PRIMARY KEY ((country, event_id), n_killed, user_id)
) WITH CLUSTERING ORDER BY (n_killed DESC, user_id ASC);

COPY top_horde_by_country_event (
    country, event_id, user_id, user_name, email, n_killed
)
FROM '/data/top_horde_by_country_event.csv'
WITH HEADER = TRUE;

"""

In [55]:
full_content = (base_content
                + content_user_by_email
                + content_dungeon_by_id
                + content_runs_by_user_dungeon
                + content_hall_of_fame_by_country_dungeon
                + content_horde_kills_by_country_event_user
                + content_top_horde_by_country_event)

In [56]:
with open("load_cassandra.cql", "w", encoding="utf-8") as f:
    f.write(full_content)

In [63]:
import subprocess

result = subprocess.run(
    'docker exec -i cassandra1 cqlsh < "C:\\Users\\adrig\\Desktop\\UPM\\3.2\\BASES DE DATOS 2\\PRACTICA 1\\Practica_bbdd\\Parte_cassandra\\load_cassandra.cql"',
    shell=True,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Using 15 child processes

Starting copy of videogame.user_by_email with columns [email, user_id, user_name, country].
Processed: 0 rows; Rate:       0 rows/s; Avg. rate:       0 rows/s

0 rows imported from 1 files in 0 day, 0 hour, 1 minutes, and 35.194 seconds (0 skipped).
Using 15 child processes

Starting copy of videogame.dungeon_by_id with columns [dungeon_id, dungeon_name].
OSError: [Errno 9] Bad file descriptor
Processed: 0 rows; Rate:       0 rows/s; Avg. rate:       0 rows/s

0 rows imported from 1 files in 0 day, 0 hour, 1 minutes, and 11.533 seconds (0 skipped).
Using 15 child processes

Starting copy of videogame.runs_by_user_dungeon with columns [user_id, dungeon_id, time_minutes, date].
Processed: 0 rows; Rate:       0 rows/s; Avg. rate:       0 rows/s

0 rows imported from 0 files in 0 day, 0 hour, 1 minutes, and 21.173 seconds (0 skipped).
Using 15 child processes

Starting copy of videogame.hall_of_fame_by_country_dungeon with columns [country, dungeon_id, dungeon_nam

In [64]:
print(result.returncode)

2


esto significa que ha ido mal, vamos a ver que estan todos los csv y la stablas

In [65]:
result = subprocess.run(
    'docker exec cassandra1 ls /data',
    shell=True,
    capture_output=True,
    text=True)

print(result.stdout)
print(result.stderr)
print(result.returncode)

dungeon_by_id.csv
hall_of_fame_by_country_dungeon.csv
horde_kills_by_country_event_user.csv
runs_by_user_dungeon.csv
top_horde_by_country_event.csv
user_by_email.csv


0


In [66]:
result = subprocess.run(
    'docker exec cassandra1 cqlsh -e "USE videogame; DESCRIBE TABLES;"',
    shell=True,
    capture_output=True,
    text=True)

print(result.stdout)
print(result.stderr)
print(result.returncode)


Connection error: ('Unable to connect to any servers', {'127.0.0.1:9042': OperationTimedOut('errors=Timed out creating connection (5 seconds), last_host=None')})

1


vale ha sido un error de conexion, comprobemos que el contenedor sigue vivo

In [67]:
result = subprocess.run(
    'docker ps',
    shell=True,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print(result.returncode)

CONTAINER ID   IMAGE           COMMAND                  CREATED          STATUS                      PORTS                                         NAMES
64178761632f   cassandra:5.0   "docker-entrypoint.sâ€¦"   55 minutes ago   Up 53 minutes (unhealthy)   0.0.0.0:9044->9042/tcp, [::]:9044->9042/tcp   cassandra3
a4359c68bc28   cassandra:5.0   "docker-entrypoint.sâ€¦"   55 minutes ago   Up 53 minutes (unhealthy)   0.0.0.0:9043->9042/tcp, [::]:9043->9042/tcp   cassandra2
6b6776945b08   cassandra:5.0   "docker-entrypoint.sâ€¦"   55 minutes ago   Up 55 minutes (unhealthy)   0.0.0.0:9042->9042/tcp, [::]:9042->9042/tcp   cassandra1


0


todo en orden, veamos el estado del cluster

In [68]:
result = subprocess.run(
    'docker exec cassandra1 nodetool status',
    shell=True,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print(result.returncode)


nodetool: Failed to connect to '127.0.0.1:7199' - SocketTimeoutException: 'Read timed out'.

1


vamos a hacer un docker restart pero ya esto desde la terminal orque en VSCode tarda mucho para luego ver que da error

```powershell

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker compose restart
[+] restart 0/3
 - Container cassandra3 Restarting                                                                                                                      2.7s
 - Container cassandra1 Restarting                                                                                                                      2.7s
 - Container cassandra2 Restarting                                                                                                                      2.7s
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 nodetool status
Datacenter: DC1
===============
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns (effective)  Host ID                               Rack
UN  172.22.0.2  4.95 MiB  128     68.6%             05880a43-c804-4a65-b5c2-b696c16d34d7  RAC1
UN  172.22.0.3  4.42 MiB  128     70.3%             3a67c8ec-39d8-4dc2-9036-6e54a0c69155  RAC1
UN  172.22.0.4  4.28 MiB  128     61.2%             dec68c37-de4e-4b57-98e9-bd01c843375d  RAC1

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "DESCRIBE KEYSPACES;"

system       system_distributed  system_traces  system_virtual_schema
system_auth  system_schema       system_views   videogame

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; DESCRIBE TABLES;"

dungeon_by_id                    runs_by_user_dungeon
hall_of_fame_by_country_dungeon  user_by_email

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; CREATE TABLE IF NOT EXISTS horde_kills_by_country_event_user (country text, event_id int, user_id int, email text, user_name text, n_killed int, PRIMARY KEY ((country, event_id), user_id));"
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; CREATE TABLE IF NOT EXISTS top_horde_by_country_event (country text, event_id int, user_id int, user_name text, email text, n_killed int, PRIMARY KEY ((country, event_id), n_killed, user_id)) WITH CLUSTERING ORDER BY (n_killed DESC, user_id ASC);"
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; DESCRIBE TABLES;"

dungeon_by_id                      runs_by_user_dungeon
hall_of_fame_by_country_dungeon    top_horde_by_country_event
horde_kills_by_country_event_user  user_by_email

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; COPY horde_kills_by_country_event_user (country, event_id, user_id, email, user_name, n_killed) FROM '/data/horde_kills_by_country_event_user.csv' WITH HEADER = TRUE;"
Using 15 child processes

Starting copy of videogame.horde_kills_by_country_event_user with columns [country, event_id, user_id, email, user_name, n_killed].
Processed: 65081 rows; Rate:   26898 rows/s; Avg. rate:   22231 rows/s
65081 rows imported from 1 files in 0 day, 0 hour, 0 minute, and 2.928 seconds (0 skipped).
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; COPY top_horde_by_country_event (country, event_id, user_id, user_name, email, n_killed) FROM '/data/top_horde_by_country_event.csv' WITH HEADER = TRUE;"

Using 15 child processes

Starting copy of videogame.top_horde_by_country_event with columns [country, event_id, user_id, user_name, email, n_killed].
Processed: 65081 rows; Rate:   19556 rows/s; Avg. rate:   22800 rows/s
65081 rows imported from 1 files in 0 day, 0 hour, 0 minute, and 2.858 seconds (0 skipped).
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM horde_kills_by_country_event_user LIMIT 5;"

 country | event_id | user_id | email                       | n_killed | user_name
---------+----------+---------+-----------------------------+----------+---------------
   fr_FR |       16 |       5 |          aaubry@example.net |       21 |   catherine30
   fr_FR |       16 |      46 |      adelaide68@example.com |       29 | etiennedufour
   fr_FR |       16 |      47 |      adelaide87@example.com |       14 | didierbernard
   fr_FR |       16 |      48 | adelaidebarbier@example.com |       23 |  anaistessier
   fr_FR |       16 |      51 |         adele81@example.org |       24 |   francoise79

(5 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM top_horde_by_country_event LIMIT 5;"

 country | event_id | n_killed | user_id | email                     | user_name
---------+----------+----------+---------+---------------------------+-----------------
   fr_FR |       16 |       39 |    3745 |    jeanevrard@example.org |    maurysusanne
   fr_FR |       16 |       39 |    9700 |     zbousquet@example.com |  delormesuzanne
   fr_FR |       16 |       38 |    5558 |  michellemahe@example.org |     henriette75
   fr_FR |       16 |       36 |    6366 | olivierperret@example.com |  rogerbourgeois
   fr_FR |       16 |       35 |     455 |      arthur05@example.com | charlotterobert

(5 rows)

funciona ya todo correctamente. comprobacion rapida de que todas las tablas están en orden:


```powershell

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; DESCRIBE TABLES;"

dungeon_by_id                      runs_by_user_dungeon
hall_of_fame_by_country_dungeon    top_horde_by_country_event
horde_kills_by_country_event_user  user_by_email

PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM user_by_email LIMIT 3;"

 email                         | country | user_id | user_name
-------------------------------+---------+---------+------------------
 caldeiraana-laura@example.net |   pt_BR |     956 |           gramos
              dito@example.net |   ja_JP |    1658 |         shohei65
     hda-conceicao@example.com |   pt_BR |    3178 | fernandesgustavo

(3 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM dungeon_by_id LIMIT 3;"

 dungeon_id | dungeon_name
------------+--------------------------------------------
          5 |     Fenwater, Hobble of the Fierce Furries
         10 | Corriedal, Sewers of the Terrible Emperors
         16 |  Dalshady, Cave of the Old-Fashioned Otaku


PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM hall_of_fame_by_country_dungeon LIMIT 3;"

 country | dungeon_id | time_minutes | date                            | email                    | dungeon_name                          | user_name
---------+------------+--------------+---------------------------------+--------------------------+---------------------------------------+---------------------
   de_DE |         15 |            0 | 2022-12-16 13:25:37.000000+0000 |   gerhardt53@example.com | Corrieclock, Pit of the Sexy Unknowns |          slobodan75
   de_DE |         15 |            1 | 2012-08-20 15:44:34.000000+0000 | franjostumpf@example.com | Corrieclock, Pit of the Sexy Unknowns | roemerklaus-guenter
   de_DE |         15 |            1 | 2013-10-11 11:00:31.000000+0000 |  gordonbeyer@example.net | Corrieclock, Pit of the Sexy Unknowns |        bolnbachigor

(3 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM horde_kills_by_country_event_user LIMIT 3;"

 country | event_id | user_id | email                  | n_killed | user_name
---------+----------+---------+------------------------+----------+---------------
   fr_FR |       16 |       5 |     aaubry@example.net |       21 |   catherine30
   fr_FR |       16 |      46 | adelaide68@example.com |       29 | etiennedufour
   fr_FR |       16 |      47 | adelaide87@example.com |       14 | didierbernard

(3 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM top_horde_by_country_event LIMIT 3;"

 country | event_id | n_killed | user_id | email                    | user_name
---------+----------+----------+---------+--------------------------+----------------
   fr_FR |       16 |       39 |    3745 |   jeanevrard@example.org |   maurysusanne
   fr_FR |       16 |       39 |    9700 |    zbousquet@example.com | delormesuzanne
   fr_FR |       16 |       38 |    5558 | michellemahe@example.org |    henriette75

(3 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh -e "USE videogame; SELECT * FROM runs_by_user_dungeon LIMIT 3;"

 user_id | dungeon_id | time_minutes | date
---------+------------+--------------+---------------------------------
    2658 |          5 |            7 | 2021-03-19 04:11:13.000000+0000
    2658 |          5 |           11 | 2018-05-02 01:10:54.000000+0000
    2658 |          5 |           14 | 2017-12-19 21:15:52.000000+0000

(3 rows)

La carga de la tabla runs_by_user_dungeon requirió ajustar los parámetros de COPY FROM, ya que con la configuración por defecto el clúster local se saturaba y producía errores de timeout. Finalmente se realizó la importación limitando el paralelismo (NUMPROCESSES = 1) y reduciendo el tamaño de los lotes, completándose correctamente la carga de 2.000.000 de filas.

```powershell 


PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra\cassandra_cluster> docker exec cassandra1 cqlsh --request-timeout=60 -e "USE videogame; COPY runs_by_user_dungeon (user_id, dungeon_id, time_minutes, date) FROM '/data/runs_by_user_dungeon.csv' WITH HEADER = TRUE AND NUMPROCESSES = 1 AND CHUNKSIZE = 100 AND MAXBATCHSIZE = 2 AND INGESTRATE = 200;"
Using 1 child processes

Processed: 2000000 rows; Rate:     133 rows/s; Avg. rate:     186 rows/s
2000000 rows imported from 1 files in 0 day, 2 hours, 59 minutes, and 9.173 seconds (0 skipped).

---

## 5. [OPCIONAL] Si el diseño lo necesita, actualiza la tabla de escrituras para incluir cualquier modificación que sea necesaria en la información que se le debe proporcionar al servidor. 

Sí, el diseño podría mejorarse ampliando la información que el juego envía al servidor en las peticiones de escritura, aunque no lo hemos considerado estrictamente necesario para que funcione. En nuestro caso, la escritura `User finish dungeon` podría incluir además de `dungeon_id`, `email`, `time_minutes` y `date`, también `user_id`, `user_name` y `country`; y la escritura `User kills monster during Horde event` podría incorporar igualmente `user_id`, `user_name` y `country` junto con `event_id`, `email` y `monster_id`. Esta ampliación encaja bien con nuestro diseño desnormalizado en Cassandra, ya que permitiría actualizar directamente las tablas de lectura sin necesidad de consultas adicionales para recuperar datos del usuario, simplificando así la lógica del servidor y haciendo más eficientes las escrituras.


In [ ]:
#!pip install cassandra-driver


   ---------------------------------------- 0/2 [geomet]
   ---------------------------------------- 0/2 [geomet]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   -------------------- ------------------- 1/2 [cassandra-driver]
   ---------------------------------------- 2/2 [cassandra-driver]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from cassandra.cluster import Cluster

cluster = Cluster(["127.0.0.1"])
session = cluster.connect("videogame")

def finish_dungeon(dungeon_id, email, time_minutes, date):
    # 1. sacar datos del usuario desde user_by_email
    row = session.execute(
        """
        SELECT user_id, user_name, country
        FROM user_by_email
        WHERE email = %s
        """,
        [email]
    ).one()

    if row is None:
        raise ValueError("usuario no encontrado")

    user_id = row.user_id
    user_name = row.user_name
    country = row.country

    # 2. sacar nombre de la dungeon desde dungeon_by_id
    dungeon_row = session.execute(
        """
        SELECT dungeon_name
        FROM dungeon_by_id
        WHERE dungeon_id = %s
        """,
        [dungeon_id]
    ).one()

    if dungeon_row is None:
        raise ValueError("dungeon no encontrada")

    dungeon_name = dungeon_row.dungeon_name

    # 3. insertar en runs_by_user_dungeon
    session.execute(
        """
        INSERT INTO runs_by_user_dungeon (user_id, dungeon_id, time_minutes, date)
        VALUES (%s, %s, %s, %s)
        """,
        [user_id, dungeon_id, time_minutes, date]
    )

    # 4. insertar en hall_of_fame_by_country_dungeon
    session.execute(
        """
        INSERT INTO hall_of_fame_by_country_dungeon
        (country, dungeon_id, dungeon_name, email, user_name, time_minutes, date)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """,
        [country, dungeon_id, dungeon_name, email, user_name, time_minutes, date]
    )

In [11]:
finish_dungeon(
    dungeon_id=7,
    email="adolf55@example.com",
    time_minutes=18,
    date="2026-03-11 19:42:00")

In [24]:
def kill_monster_in_horde(event_id, email, n_killed_total):
    # 1. resolver datos del usuario
    row = session.execute(
        """
        SELECT user_id, user_name, country
        FROM user_by_email
        WHERE email = %s
        """,
        [email]
    ).one()

    if row is None:
        raise ValueError("usuario no encontrado")

    user_id = row.user_id
    user_name = row.user_name
    country = row.country

    # 2. actualizar tabla acumulada
    session.execute(
        """
        INSERT INTO horde_kills_by_country_event_user
        (country, event_id, user_id, email, user_name, n_killed)
        VALUES (%s, %s, %s, %s, %s, %s)
        """,
        [country, event_id, user_id, email, user_name, n_killed_total]
    )

    # 3. actualizar tabla de ranking
    session.execute(
        """
        INSERT INTO top_horde_by_country_event
        (country, event_id, n_killed, user_id, user_name, email)
        VALUES (%s, %s, %s, %s, %s, %s)
        """,
        [country, event_id, n_killed_total, user_id, user_name, email]
    )

In [46]:
kill_monster_in_horde(
    event_id=15,
    email="adolf55@example.com",
    n_killed_total=90)

In [43]:
row = session.execute("""
    SELECT email, user_id, user_name, country
    FROM user_by_email
    WHERE email = %s
""", ["adolf55@example.com"]).one()

print(row)

Row(email='adolf55@example.com', user_id=57, user_name='khettner', country='de_DE')


In [47]:
rows = session.execute("""
    SELECT country, event_id, user_id, email, user_name, n_killed
    FROM horde_kills_by_country_event_user
    WHERE country = %s AND event_id = %s
""", ["de_DE", 15])

for r in rows:
    if r.email == "adolf55@example.com":
        print(r)

Row(country='de_DE', event_id=15, user_id=57, email='adolf55@example.com', user_name='khettner', n_killed=90)


In [48]:
queries_reading = """
USE videogame;

-- 1. HALL OF FAME
CONSISTENCY LOCAL_ONE;

SELECT dungeon_id, dungeon_name, email, user_name, time_minutes, date
FROM hall_of_fame_by_country_dungeon
WHERE country = 'fr_FR' AND dungeon_id = 7
LIMIT 5;

-- 2. USER STATISTICS
CONSISTENCY LOCAL_ONE;

SELECT time_minutes, date
FROM runs_by_user_dungeon
WHERE user_id = 123 AND dungeon_id = 7;

-- 3. TOP HORDE
CONSISTENCY LOCAL_QUORUM;

SELECT user_id, user_name, email, n_killed
FROM top_horde_by_country_event
WHERE country = 'fr_FR' AND event_id = 15
LIMIT 10;
"""
queries_writing= """


-- 4. USER FINISH DUNGEON
CONSISTENCY LOCAL_ONE;

INSERT INTO runs_by_user_dungeon (user_id, dungeon_id, time_minutes, date)
VALUES (123, 7, 18, '2026-03-11 19:42:00');

CONSISTENCY LOCAL_ONE;

INSERT INTO hall_of_fame_by_country_dungeon
(country, dungeon_id, dungeon_name, email, user_name, time_minutes, date)
VALUES ('fr_FR', 7, 'Dungeon Example', 'laura.martin@example.com', 'laura23', 18, '2026-03-11 19:42:00');

-- 5. USER KILLS MONSTER DURING HORDE EVENT
CONSISTENCY LOCAL_QUORUM;

INSERT INTO horde_kills_by_country_event_user
(country, event_id, user_id, email, user_name, n_killed)
VALUES ('fr_FR', 15, 123, 'laura.martin@example.com', 'laura23', 43);

CONSISTENCY LOCAL_QUORUM;

INSERT INTO top_horde_by_country_event
(country, event_id, n_killed, user_id, user_name, email)
VALUES ('fr_FR', 15, 43, 123, 'laura23', 'laura.martin@example.com');
"""

In [49]:
queri_total = queries_reading + "\n" + queries_writing

In [50]:
with open("queries_videogame.cql", "w", encoding="utf-8") as f:
    f.write(queri_total)

comprobamos que funciona

```powershell 
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra> docker exec cassandra1 cqlsh -e "USE videogame; SELECT email, user_id, user_name, country FROM user_by_email LIMIT 5;"

 email                         | user_id | user_name        | country
-------------------------------+---------+------------------+---------
 caldeiraana-laura@example.net |     956 |           gramos |   pt_BR
              dito@example.net |    1658 |         shohei65 |   ja_JP
     hda-conceicao@example.com |    3178 | fernandesgustavo |   pt_BR
                bo@example.com |     798 |          jaehogu |   ko_KR
           mingqin@example.com |    5659 |            minyi |   zh_CN

(5 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra> docker exec cassandra1 cqlsh -e "USE videogame; SELECT dungeon_id, dungeon_name FROM dungeon_by_id LIMIT 5;"

 dungeon_id | dungeon_name
------------+-------------------------------------------------------
          5 |                Fenwater, Hobble of the Fierce Furries
         10 |            Corriedal, Sewers of the Terrible Emperors
         16 |             Dalshady, Cave of the Old-Fashioned Otaku
         13 | Greatcliffe, Castle of the Magnificent Sumo Wrestlers
         11 |         Bentalaun, Vault of the Old-Fashioned Pirates

(5 rows)
PS C:\Users\adrig\Desktop\UPM\3.2\BASES DE DATOS 2\PRACTICA 1\Practica_bbdd\Parte_cassandra> docker exec cassandra1 cqlsh -e "USE videogame; SELECT country, event_id, user_id, user_name, email, n_killed FROM top_horde_by_country_event LIMIT 5;"

 country | event_id | user_id | user_name      | email                    | n_killed
---------+----------+---------+----------------+--------------------------+----------
   de_DE |       15 |      57 |       khettner |      adolf55@example.com |       90
   de_DE |       15 |      57 |       khettner |      adolf55@example.com |       43
   fr_FR |       16 |    3745 |   maurysusanne |   jeanevrard@example.org |       39
   fr_FR |       16 |    9700 | delormesuzanne |    zbousquet@example.com |       39
   fr_FR |       16 |    5558 |    henriette75 | michellemahe@example.org |       38

(5 rows)